In [1]:
import json

with open("chunks_v2.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)

print(f"Loaded {len(chunks)} chunks")

Loaded 168 chunks


In [2]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer("all-MiniLM-L6-v2")
print("Model loaded")

c:\Users\LENOVO\Desktop\RAG-handbook\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2830.87it/s]


Model loaded


In [3]:
texts = [chunk["text"] for chunk in chunks]
embeddings = model.encode(texts, show_progress_bar=True)
print(embeddings.shape)

Batches: 100%|██████████| 6/6 [00:01<00:00,  4.41it/s]

(168, 384)


In [4]:
import numpy as np

np.save("embeddings.npy", embeddings)
print("Saved embeddings.npy")

Saved embeddings.npy


In [5]:
def search(questions, top_k=5):
  question_embedding = model.encode(questions)
  scores = util.cos_sim(question_embedding, embeddings)[0]
  top_results = scores.argsort(descending=True)[:top_k]

  for rank, idx in enumerate(top_results):
    idx = int(idx)
    print(f"Rank: {rank+1}")
    print(f"ID: {chunks[idx]['id']}")
    print(f"Score: {scores[idx]:.4f}")
    print(f"Section: {chunks[idx]['section']}")
    print(f"Text: {chunks[idx]['text'][:130]}...")
    print()

In [6]:
search("What is the maximum duration to complete the Bachelor's degree?")

Rank: 1
ID: 2
Score: 0.7266
Section: ACADEMIC REGULATIONS
Text: 1.With due consideration to the maximum duration for the award of the Bachelor’s Degree, the Bachelor's Degree student should not ...

Rank: 2
ID: 6
Score: 0.7124
Section: ACADEMIC REGULATIONS
Text: 5.A student registered for the Bachelor’s Degree shall be placed at 2nd, 3rd,4th year level if he/she has successfully completed n...

Rank: 3
ID: 38
Score: 0.6336
Section: Rules and Regulations of Attendance
Text: computing the maximum permitted period for graduation....

Rank: 4
ID: 1
Score: 0.6193
Section: ACADEMIC REGULATIONS
Text: Course load B.Sc. (Bachelor's Degree) in Nursing requires 134 credit hours, to be completed in an average duration of four years a...

Rank: 5
ID: 76
Score: 0.6017
Section: Rules and Regulations of Attendance
Text: 7. The graduation certificate is awarded upon the fulfillment of all the requirements at the end of every semester. However, the g...



In [7]:
eval_set = [
    {"question": "How many credit hours are required for the B.Sc. in Nursing?", "correct_id": 1},
    {"question": "What is the maximum duration to complete the Bachelor's degree?", "correct_id": 1},
    {"question": "What is the minimum and maximum credit hours I can register in a regular semester?", "correct_id": 2},
    {"question": "How many credit hours can I register in summer?", "correct_id": 5},
    {"question": "What percentage of classes can I miss before being barred from the exam?", "correct_id": 9},
    {"question": "What happens if my absence exceeds 20% due to an accepted excuse?", "correct_id": 11},
    {"question": "How soon after enrollment can a newly enrolled student postpone their studies?", "correct_id": 30},
    {"question": "How is the final grade of a course calculated?", "correct_id": 40},
    {"question": "What does nursing ethics refer to?", "correct_id": 90},
    {"question": "How should a request to change specialization be submitted?", "correct_id": 21},
    {"question": "Who decides on requests to change specialization, and what is it subject to?", "correct_id": 20},
    {"question": "Does the semester following a dismissal count toward the maximum graduation period?", "correct_id": 35},
    {"question": "Can Arab students change to a specialization with a lower admission average?", "correct_id": 25},
]


In [8]:
def get_top_k_ids(question, top_k=5):
    question_embedding = model.encode(question)
    scores = util.cos_sim(question_embedding, embeddings)[0]
    top_results = scores.argsort(descending=True)[:top_k]
    return [int(chunks[i]['id']) for i in top_results]

In [9]:
hits_at_1, hits_at_3, hits_at_5 = 0, 0, 0

for item in eval_set:
    retrieved_ids = get_top_k_ids(item["question"], top_k=5)
    correct = item["correct_id"]
    in_top1 = correct in retrieved_ids[:1]
    in_top3 = correct in retrieved_ids[:3]
    in_top5 = correct in retrieved_ids[:5]
    hits_at_1 += in_top1
    hits_at_3 += in_top3
    hits_at_5 += in_top5
    print(f"Q: {item['question']}")
    print(f"Expected ID: {correct} | Retrieved IDs: {retrieved_ids}")
    print(f"Top1: {'✓' if in_top1 else '✗'}  Top3: {'✓' if in_top3 else '✗'}  Top5: {'✓' if in_top5 else '✗'}\n")

n = len(eval_set)
print(f"Recall@1: {hits_at_1/n:.2f}  ({hits_at_1}/{n})")
print(f"Recall@3: {hits_at_3/n:.2f}  ({hits_at_3}/{n})")
print(f"Recall@5: {hits_at_5/n:.2f}  ({hits_at_5}/{n})")

Q: How many credit hours are required for the B.Sc. in Nursing?
Expected ID: 1 | Retrieved IDs: [1, 63, 6, 2, 4]
Top1: ✓  Top3: ✓  Top5: ✓

Q: What is the maximum duration to complete the Bachelor's degree?
Expected ID: 1 | Retrieved IDs: [2, 6, 38, 1, 76]
Top1: ✗  Top3: ✗  Top5: ✓

Q: What is the minimum and maximum credit hours I can register in a regular semester?
Expected ID: 2 | Retrieved IDs: [4, 5, 2, 61, 65]
Top1: ✗  Top3: ✓  Top5: ✓

Q: How many credit hours can I register in summer?
Expected ID: 5 | Retrieved IDs: [5, 4, 59, 74, 61]
Top1: ✓  Top3: ✓  Top5: ✓

Q: What percentage of classes can I miss before being barred from the exam?
Expected ID: 9 | Retrieved IDs: [9, 11, 126, 10, 46]
Top1: ✓  Top3: ✓  Top5: ✓

Q: What happens if my absence exceeds 20% due to an accepted excuse?
Expected ID: 11 | Retrieved IDs: [118, 48, 12, 117, 11]
Top1: ✗  Top3: ✗  Top5: ✓

Q: How soon after enrollment can a newly enrolled student postpone their studies?
Expected ID: 30 | Retrieved IDs: [

loading the API code safely :

In [10]:
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")

print("Key loaded:", api_key is not None)

Key loaded: True


configuring the Gemini client :
didn't actually call gemini just prepared it 

In [16]:
#import google.generativeai as genai

#genai.configure(api_key=api_key)
#llm = genai.GenerativeModel("gemini-3.8-flash")

#print("Gemini ready")


import google.generativeai as genai

genai.configure(api_key=api_key)
llm = genai.GenerativeModel("gemini-3.5-flash-lite")

print("Gemini ready")

Gemini ready


The function tying retrieval and generation together 
— the actual RAG step

In [17]:
def answer_question(question, top_k=5):
    retrieved_ids = get_top_k_ids(question, top_k=top_k)
    retrieved_chunks = [c for c in chunks if c["id"] in retrieved_ids]

    context = "\n\n".join(
        f"[Chunk {c['id']}] {c['text']}" for c in retrieved_chunks
    )

    prompt = f"""You are answering questions about the JUST Faculty of Nursing Student Handbook.
Answer the question using ONLY the information in the passages below.
If the passages don't contain the answer, say "I don't have enough information to answer that."
Cite the chunk ID(s) you used in your answer.

Passages:
{context}

Question: {question}

Answer:"""

    response = llm.generate_content(prompt)
    return response.text

First single test:

In [18]:
answer = answer_question("What is the maximum duration to complete the Bachelor's degree?")
print(answer)

The maximum duration to complete the Bachelor's degree in Nursing is six years [Chunk 1].


Testing generation across multiple questions, with rate-limit handling

In [14]:
import time

for item in eval_set[:6]:
    print(f"Q: {item['question']}")
    try:
        answer = answer_question(item['question'])
        print(f"A: {answer}")
        has_citation = "[Chunk" in answer
        print(f"Citation present: {'✓' if has_citation else '✗'}")
    except Exception as e:
        print(f"Request failed (likely rate limit): {e}")
    print()
    time.sleep(10)

Q: How many credit hours are required for the B.Sc. in Nursing?
Request failed (likely rate limit): 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash
Please retry in 1.241793933s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerDayPerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-3.8-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 20
}
, retry_delay {
  seconds: 1
}
]

Q: What is

the interface lololoeeesh

In [19]:
#import gradio as gr
#def gradio_answer(question):
 #   if not question.strip():
  #      return "Please enter a question "
   # try:  
    #    return answer_question(question)
    #except Exception as e :
     #   return "something went wrong: " + str(e)

#demo=gr.Interface(
 #   fn=gradio_answer,
  #  inputs=gr.Textbox(label="Ask about the JUST Nursing Student Handbook", placeholder="e.g. How many credit hours can I register in summer?"),
   # outputs=gr.Textbox(label="Answer"),
    #title="JUST Nursing Handbook Q&A",
    #description="Ask a question about academic regulations, attendance, or clinical guidelines . Answers are grounded in the handbook and cite the source chunk."
#)

#demo.launch()

import gradio as gr
import time

def gradio_answer(question):
    if not question.strip():
        return "Please enter a question."
    for attempt in range(3):
        try:
            return answer_question(question)
        except Exception as e:
            if "429" in str(e) and attempt < 2:
                time.sleep(15)
                continue
            return f"Something went wrong: {e}"

demo=gr.Interface(
    fn=gradio_answer,
    inputs=gr.Textbox(label="Ask about the JUST Nursing Student Handbook", placeholder="e.g. How many credit hours can I register in summer?"),
    outputs=gr.Textbox(label="Answer"),
    title="JUST Nursing Handbook Q&A",
    description="Ask a question about academic regulations, attendance, or clinical guidelines . Answers are grounded in the handbook and cite the source chunk."
)

demo.launch()


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


gradio_answer(question) — a small wrapper around your existing answer_question(). It's not new logic — it just adds two safety checks a real user interacting with a text box actually needs: an empty input (someone clicks the button without typing anything), and a graceful message if the API call fails (like the rate limit we just saw), instead of the whole interface crashing.
gr.Interface(...) — Gradio's main building block. fn is which function runs when the user submits something. inputs/outputs define what the textbox looks like on each side. title/description are just the page's header text.
demo.launch() — starts a local web server and gives you a clickable link (usually http://127.0.0.1:7860) that opens the actual interface in your browser.